In [1]:
import os
os.chdir("/workspaces/pythonProject/medallion_v2/data/notebooks")

# 03 · Load the five customer sources into Turso (customer360 scenario)
Creates five source tables in Turso (`crm_accounts`, `billing_customers`, `support_contacts`, `web_visitors`, `marketing_leads`) and loads one CSV batch of each (upsert, so re-applying a batch is safe).

* **batch 1**: initial load, overlapping on email, with conflicting values, duplicates and bad rows
* **batch 2**: plan changes, cancellations, new support contacts, more visits, and four leads that convert into CRM accounts and paying customers

Needs `00_config.ipynb` (already in your project). Do **not** re-run batch 1 after batch 2 unless you reset first (`RESET_SOURCE=1` drops this domain's tables in Turso and starts over).

In [2]:
import os
BATCH = int(os.getenv("BATCH", 1))              # 1 or 2: change the number, then run
RESET = os.getenv("RESET_SOURCE") == "1"        # True = start over for this domain

In [3]:
%run -i ./00_config.ipynb

rule library: 27 checks for 9 entities


In [4]:
open_warehouse()
print("connected to Turso")

C360 = load_domain("customer360")                                      # tables and keys come from domains/customer360.yml
TABLES = {e: ENTITIES[e]["keys"][0] for e in C360["entities"]}

if RESET: reset_domain(C360)

ddl = "\n".join(l for l in find_file("schema_multisource.sql").read_text().splitlines() if not l.strip().startswith("--"))
for stmt in [s.strip() for s in ddl.split(";") if s.strip()]:
    con.execute(stmt)
con.commit()

connected to Turso
domain 'customer360': 5 entities, 1 dimensions, 0 facts


In [5]:
for table in TABLES:
    try:
        f = find_file(f"{table}_batch{BATCH}.csv")
    except FileNotFoundError:
        continue
    n, secs = upsert_csv(table, f, TABLES[table])
    print(f"{table:18s} loaded {n:4d} rows from {f.name} in {secs:.1f}s")

for t in TABLES:
    print(t, con.one(f"SELECT count(*), max(updated_at) FROM {t}"))

crm_accounts       loaded   60 rows from crm_accounts_batch1.csv in 0.1s
billing_customers  loaded   47 rows from billing_customers_batch1.csv in 0.1s
support_contacts   loaded   34 rows from support_contacts_batch1.csv in 0.1s
web_visitors       loaded   31 rows from web_visitors_batch1.csv in 0.1s
marketing_leads    loaded   35 rows from marketing_leads_batch1.csv in 0.1s
crm_accounts (60, '2026-10-30 14:13:55')
billing_customers (47, '2026-10-31 20:29:53')
support_contacts (34, '2026-10-31 21:16:00')
web_visitors (31, '2026-10-31 18:55:13')
marketing_leads (35, '2026-10-28 20:20:37')


In [6]:
release_all()          # closes the Turso connection (the next notebook opens its own)